In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

g = torch.Generator().manual_seed(2147483647)
random.seed(42)

In [ ]:
names = open('names.txt').read().splitlines()
len(names)

In [ ]:
chars = sorted(list(set(''.join(names))))

c_to_i = {c:i+1 for i,c in enumerate(chars)}
c_to_i['.'] = 0

i_to_c = {i:c for c,i in c_to_i.items()}

In [474]:
def build_dataset(words):
    X, Y = [], []
    block_size = 3

    for n in words:
        context = [0] * block_size

        for ch in n + '.':
            ix = c_to_i[ch]
            Y.append(ix)
            X.append(context)

            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)

    return X, Y

random.shuffle(names)

n1 = int(0.8 * len(names))
n2 = int(0.9 * len(names))

# Splitting data into 80 (training) / 10 (dev) / 10 (testing)

X_train, Y_train = build_dataset(names[:n1])
X_dev, Y_dev = build_dataset(names[n1:n2])
X_test, Y_test = build_dataset(names[n2:])

In [479]:
C = torch.randn((27, 10), generator=g)
W1 = torch.randn((30, 300), generator=g)
b1 = torch.randn(300, generator=g)
W2 = torch.randn((300, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [480]:
for p in parameters:
    p.requires_grad = True

sum(p.nelement() for p in parameters)

17697

In [439]:
lre = torch.linspace(-3, 0, 1000)
lr = 10**lre

In [497]:
lri = []
lossi = []

for i in range(30000):
    ix = torch.randint(0, X_train.shape[0], (32,))
    emb = C[X_train[ix]]
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y_train[ix])
    print(loss.item())

    # Backpropagation
    for p in parameters:
        p.grad = None
    loss.backward()

    # Update Step
    for p in parameters:
        p.data += -0.0001 * p.grad

2.492875337600708
2.589853286743164
2.0263569355010986
2.0644755363464355
1.9364341497421265
2.001746892929077
1.971266269683838
2.320873737335205
2.629307746887207
2.10817289352417
2.1667025089263916
2.452223539352417
2.37770414352417
1.8168532848358154
2.4198827743530273
2.151362180709839
2.0795187950134277
2.1336967945098877
2.2384872436523438
1.843195915222168
2.006561279296875
2.184436559677124
2.0368587970733643
2.050243616104126
1.9448179006576538
1.7729729413986206
2.253786563873291
2.1977174282073975
2.1893210411071777
2.0913567543029785
1.8524391651153564
2.0103518962860107
2.4577014446258545
1.9881285429000854
2.1954681873321533
2.300419569015503
2.0663774013519287
2.0657622814178467
1.9044591188430786
2.0408880710601807
2.00522780418396
2.2227132320404053
2.3269717693328857
2.021491289138794
2.174712657928467
2.052490711212158
1.9070854187011719
1.5374325513839722
2.3417022228240967
2.3940348625183105
2.294367790222168
1.9320728778839111
1.888789176940918
2.02067494392395
2

In [498]:
with torch.no_grad():
    emb = C[X_dev]
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y_dev)

print(loss.item())

2.1612465381622314


In [501]:
for i in range(20):
    out = []
    context = [0] * 3

    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(i_to_c[ix])

        if ix == 0:
            break

    print(''.join(out))

robella.
deiryn.
keven.
pmanna.
uaras.
shenry.
gise.
shrelese.
raan.
milanesteferm.
colbiellyus.
madyqavavelissa.
balila.
alecidson.
wid.
natis.
nasaan.
neal.
khi.
jalynney.
